#  Intermediate Scraping: Multiple Sources
In this session, we expand our scraping toolkit to manage **lists of URLs**. This is essential when extracting data from multiple municipal websites or large archives.

---

###  Handling lists of targets
When we have more than one URL, we store them in a **List**. 
In Python, we create lists using square brackets `[]`.

## Theory: Scaling NLP through Iteration
When moving from single documents to corpora, we transition from individual variables to **Iterative Processing**. In Python, this is achieved through loops. From a computer science perspective, we are 'mapping' a set of inputs (URLs) to a set of outputs (processed text). Efficiently managing these collections is the foundation of big data analysis in linguistics.


In [ ]:
start_urls = ['https://www.comune.ortisei.bz.it/system/web/zeitung.aspx?menuonr=219371088&sprache=3',
             'https://www.comune.meltina.bz.it/de/Neuigkeiten/Gemeindeblatt_-_Die_Schronn'
             ]

###  List Iteration
We can use a `for` loop to visit every URL in our list systematically:
```python
for url in start_urls:
    # scrape each one...
```

####  Review: Requests and Responses
Remember: `requests.get()` sends an HTTP request. The `response.text` property contains the HTML code of the page, which we then feed into BeautifulSoup.

In [ ]:
url = 'https://www.comune.meltina.bz.it/de/Neuigkeiten/Gemeindeblatt_-_Die_Schronn'
import requests

####  Review: Requests and Responses
Remember: `requests.get()` sends an HTTP request. The `response.text` property contains the HTML code of the page, which we then feed into BeautifulSoup.

In [ ]:
r = requests.get(url)
r.text

<a href="/de/Neuigkeiten/Gemeindeblatt_-_Die_Schronn/Die_Schronn_6-2026" class="text-decoration-none" title="weiter zu Die Schronn 6-2026" data-focus-mouse="false"><div class="col-12 mb-0 h4">
                      <h3 class="text-break">Die Schronn 6-2026</h3>
                    </div></a>

In [ ]:
from bs4 import BeautifulSoup
source = r.text
soup = BeautifulSoup(source)

In [ ]:
for tag in soup.find_all("a", class_= "text-decoration-none"):
    print("https://www.comune.meltina.bz.it" + tag.attrs['href'])

####  Review: Requests and Responses
Remember: `requests.get()` sends an HTTP request. The `response.text` property contains the HTML code of the page, which we then feed into BeautifulSoup.

In [ ]:
page = "https://www.comune.meltina.bz.it/de/Neuigkeiten/Gemeindeblatt_-_Die_Schronn/Die_Schronn_4-2025"
rpage = requests.get(page)
soup_page = BeautifulSoup(rpage.text)


In [ ]:
soup_page.find('div', class_="card card-teaser shadow rounded")

####  Review: Requests and Responses
Remember: `requests.get()` sends an HTTP request. The `response.text` property contains the HTML code of the page, which we then feed into BeautifulSoup.

In [ ]:
# If we point requests to this page we are not going too far
"https://www.comune.meltina.bz.it/de/system/web/GetDocument.ashx?cts=1763019782&amp;fileId=1519806" title="Gemeindeblatt"

In [ ]:
from selenium import webdriver
import time
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
from selenium import webdriver

# No service or manager needed in Selenium 4.6+
DRIVER = webdriver.Chrome() 


In [ ]:
#xpath

In [ ]:
def get_pdfs(u):
    DRIVER.get(u)
    time.sleep(2)
    e = DRIVER.find_element(By.XPATH, '//a[@class="text-decoration-none piwik_download"]')
    DRIVER.execute_script("arguments[0].click();", e)
    # e.click()

get_pdfs("https://www.comune.meltina.bz.it/de/Neuigkeiten/Gemeindeblatt_-_Die_Schronn/Die_Schronn_4-2025")

Think of it as the difference between a **physical mouse click** and an **internal command.**

When you use the standard `e.click()`, Selenium tries to behave like a real human. When you use `execute_script`, you are going "under the hood" to the browser's engine.

### 1. Bypassing the "Physical" Check

The standard `e.click()` is a **high-level action**. Before Selenium clicks, it performs several checks:

* **Visibility:** Is the element hidden?
* **Interactivity:** Is there a "glass wall" (like a cookie banner or a transparent overlay) on top of it?
* **Location:** Is it actually within the clickable area of the screen?

If a cookie banner is even 1 pixel over your link, Selenium's "finger" hits the banner instead of the link and throws the `ElementClickIntercepted` error.

### 2. Direct JavaScript Execution

`DRIVER.execute_script("arguments[0].click();", e)` is a **low-level command**.

* **`execute_script`**: Tells the browser to run raw JavaScript code.
* **`arguments[0]`**: This is a placeholder for the variable `e` (your element) that you passed at the end of the line.
* **`.click()`**: This is the native JavaScript method.

**Crucially:** JavaScript doesn't care about overlays. It finds the element in the Document Object Model (DOM) and triggers its "onclick" event directly. It’s like reaching through a window to flip a switch rather than trying to push the window itself.

---

### Comparison Table

| Feature | `e.click()` (Standard) | `execute_script` (JavaScript) |
| --- | --- | --- |
| **Realism** | Mimics a real user's mouse. | Bypasses user simulation. |
| **Safety** | Won't click if something is blocking. | Clicks no matter what is on top. |
| **Scrolling** | Often requires the element to be on screen. | Works even if the element is off-screen. |
| **Reliability** | Fails often on modern, "pop-up heavy" sites. | Almost always succeeds. |

### Is there a downside?

The only "risk" is that you might click something that a real user *couldn't* actually see or interact with. If you are web scraping for data, this is usually **exactly** what you want. If you are testing a website's usability, it's "cheating" because it doesn't prove a human can click it.

Since you're trying to download PDFs from a municipal site, **cheating is the way to go!** Would you like to see how to use this same JavaScript trick to scroll smoothly to the bottom of the page?

####  Review: Requests and Responses
Remember: `requests.get()` sends an HTTP request. The `response.text` property contains the HTML code of the page, which we then feed into BeautifulSoup.

In [ ]:
import urllib.robotparser
import time

def can_i_crawl(url, user_agent="*"):
    # 1. Initialize the parser
    rp = urllib.robotparser.RobotFileParser()
    
    # 2. Set the robots.txt location
    # Most sites use: domain.com/robots.txt
    rp.set_url("https://www.comune.ortisei.bz.it/robots.txt")
    
    # 3. Read and parse the file
    try:
        rp.read()
    except Exception as e:
        print(f"Could not read robots.txt: {e}")
        return False

    # 4. Check if your specific URL is allowed
    allowed = rp.can_fetch(user_agent, url)
    
    # 5. Check for a recommended delay (Politeness)
    delay = rp.crawl_delay(user_agent)
    
    return allowed, delay

# Usage example:
target_url = "https://www.comune.ortisei.bz.it/de/Neuigkeiten"
is_allowed, wait_time = can_i_crawl(target_url)

if is_allowed:
    print(f" Access allowed to: {target_url}")
    if wait_time:
        print(f"⏱ Site requests a delay of {wait_time} seconds between requests.")
else:
    print(f" Access DISALLOWED by robots.txt for: {target_url}")

In [ ]:
type(s)

In [ ]:
a = "11 aa bb 34 65 gg 2".split()
for i in a:
    try:
        print(int(i))
    except:
        print(i, "is not a number")

In [ ]:
a = "11 aa bb 34 65 gg 2".split()
for i in a:
    if i.isdigit():
        print(int(i))
    else:
        print(i, "is not a number")

In [ ]:
t = None
type(t)

In [ ]:
if t == None:
    print('Yes')

In [ ]:
rp = urllib.robotparser.RobotFileParser()
    
    # 2. Set the robots.txt location
    # Most sites use: domain.com/robots.txt
rp.set_url("https://www.comune.ortisei.bz.it/robots.txt")
allowed = rp.can_fetch("*", "https://www.comune.ortisei.bz.it/robots.txt")
allowed

## Exercises
Complete the following tasks to practice the concepts covered in this lesson. Aim for efficient, readable code.


### Exercise 1
Create a list of 3 URLs from different news websites. Write a loop that visits each URL, extracts the main title, and stores it in a dictionary where the key is the URL and the value is the title.


### Exercise 2
Given a list of municipal URLs (like those from South Tyrol), write a loop that categorizes them into two lists: `german_sites` and `italian_sites`, based on whether the URL contains '/de/' or '/it/'.


### Exercise 3
Modify the scraping loop to count the total number of paragraphs (`<p>` tags) across all pages in your target list and print the final sum.


### Exercise 4
Write a script that visits a page and extracts all text within a specific `div` class. If the class is missing on one of the pages, use a `try-except` block to prevent the code from crashing.


### Exercise 5
Challenge: Extract all internal links from a page (links that start with '/' or the site's own domain) and save them into a set to ensure they are unique.
